# 06 - RoBERTa Fine-Tuning: Cognitive Distortion Detector

**Run this notebook on Google Colab with a GPU runtime (T4 is sufficient).**

## What this notebook does
1. Installs dependencies
2. Uploads your local data files (no Drive needed)
3. Merges `train.csv` + `synthetic_data.csv` into one training set
4. Carves a 10% validation split from the merged training set
5. Uses existing `class_weights.npy` - recomputed for merged distribution
6. Fine-tunes `roberta-base` for multi-label classification (10 distortion labels)
7. Saves best checkpoint (by val micro-F1) and evaluates on held-out `test.csv`
8. **Downloads the model as a zip directly to your computer** - no Drive space used

**Files to upload when prompted (4 files):**
- `data/processed/train.csv`
- `data/processed/test.csv`
- `data/processed/class_weights.npy`
- `data/synthetic/synthetic_data.csv`

**After training:** unzip `roberta_cognitive_distortion.zip` and place the folder at `models/roberta_cognitive_distortion/` in your local project.

## ① Install dependencies

In [ ]:
# Run once - no restart needed
!pip install -q transformers scikit-learn

## ② Upload local data files

In [ ]:
from google.colab import files

print("Upload these 4 files from your local project:")
print("  • data/processed/train.csv")
print("  • data/processed/test.csv")
print("  • data/processed/class_weights.npy")
print("  • data/synthetic/synthetic_data.csv")
print()
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

## ③ Imports & config

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

# ── Reproducibility ──────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Label set - must match src/config.py exactly ───────────────────────────────────────────
DISTORTION_LABELS = [
    "All-or-nothing thinking",
    "Overgeneralization",
    "Mental filter",
    "Should statements",
    "Labeling",
    "Personalization",
    "Catastrophising/Magnification",
    "Emotional Reasoning",
    "Mind Reading",
    "Fortune-telling",
]
NUM_LABELS = len(DISTORTION_LABELS)

# ── Hyperparameters ────────────────────────────────────────────────────────────────────────
MODEL_NAME    = "roberta-base"
MAX_LEN       = 256      # handles multi-sentence inputs (DistilBERT used 128)
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
WARMUP_RATIO  = 0.1      # 10% of total steps for linear warmup
WEIGHT_DECAY  = 0.01
THRESHOLD     = 0.40     # sigmoid threshold for positive prediction
VAL_SPLIT     = 0.10     # 10% of merged train carved out for validation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## ④ Load and merge data

In [ ]:
# ── Load original processed data ─────────────────────────────────────────────────────────────────
df_train_orig = pd.read_csv("train.csv")
df_test       = pd.read_csv("test.csv")

print(f"Original train : {len(df_train_orig)} rows")
print(f"Test (held-out): {len(df_test)} rows")

# ── Load synthetic data ──────────────────────────────────────────────────────────────────────
df_syn = pd.read_csv("synthetic_data.csv")
print(f"Synthetic data : {len(df_syn)} rows")

# ── Keep only the columns we need ──────────────────────────────────────────────────────────────
COLS = ["text"] + DISTORTION_LABELS
df_train_orig = df_train_orig[COLS].copy()
df_test       = df_test[COLS].copy()
df_syn        = df_syn[COLS].copy()

# ── Merge original train + synthetic into training pool ─────────────────────────────
df_merged = pd.concat([df_train_orig, df_syn], ignore_index=True)
df_merged["text"] = df_merged["text"].astype(str).str.strip()
df_merged.dropna(subset=["text"], inplace=True)
df_merged.drop_duplicates(subset=["text"], inplace=True)
df_merged.reset_index(drop=True, inplace=True)

print(f"\nMerged training pool (after dedup): {len(df_merged)} rows")
print("\nLabel distribution in merged pool:")
print(df_merged[DISTORTION_LABELS].sum().sort_values(ascending=False).to_string())

In [ ]:
# ── Carve out validation split from merged training pool ─────────────────────────────
# Stratify on dominant label so all labels are represented in val
label_matrix = df_merged[DISTORTION_LABELS].values
dominant = np.where(
    label_matrix.sum(axis=1) == 0,
    NUM_LABELS,                        # all-zero rows get their own group
    label_matrix.argmax(axis=1)
)

df_train_final, df_val = train_test_split(
    df_merged, test_size=VAL_SPLIT, random_state=SEED, stratify=dominant
)
df_train_final.reset_index(drop=True, inplace=True)
df_val.reset_index(drop=True, inplace=True)

print(f"Final train : {len(df_train_final)}")
print(f"Validation  : {len(df_val)}")
print(f"Test        : {len(df_test)}  (original held-out split, untouched)")

## ⑤ Class weights for loss

In [ ]:
# ── Recompute pos_weight from merged training set ──────────────────────────────────────
# We recompute rather than reusing class_weights.npy because the distribution
# shifted after adding ~2300 synthetic examples.
# pos_weight[i] = neg_count[i] / pos_count[i]  - penalises missing a positive more
label_matrix_train = df_train_final[DISTORTION_LABELS].values.astype(np.float32)
pos_counts = label_matrix_train.sum(axis=0)
neg_counts = len(label_matrix_train) - pos_counts
pos_weight = torch.tensor(
    neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32
).to(DEVICE)

# Load original weights for comparison
orig_cw = np.load("class_weights.npy")

print(f"{'Label':<35} {'orig_weight':>12} {'new_pos_weight':>15}")
print("-" * 65)
for label, ow, nw in zip(DISTORTION_LABELS, orig_cw, pos_weight.cpu().numpy()):
    print(f"{label:<35} {ow:>12.4f} {nw:>15.4f}")

## ⑥ Dataset & DataLoaders

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)


class DistortionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts     = df["text"].tolist()
        self.labels    = df[DISTORTION_LABELS].values.astype(np.float32)  # (N, 10)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels"        : torch.tensor(self.labels[idx], dtype=torch.float32),
        }


train_ds = DistortionDataset(df_train_final, tokenizer)
val_ds   = DistortionDataset(df_val,         tokenizer)
test_ds  = DistortionDataset(df_test,        tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

## ⑦ Model, loss, optimizer, scheduler

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
model.to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ── Loss: BCEWithLogitsLoss with per-label pos_weight ──────────────────────────────────
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Optimizer: AdamW, no weight decay on bias/LayerNorm ──────────────────────────────
no_decay = ["bias", "LayerNorm.weight"]
optimizer = torch.optim.AdamW(
    [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

# ── Linear warmup scheduler ───────────────────────────────────────────────────────────────────
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")

## ⑧ Training loop

In [ ]:
def evaluate(model, loader):
    """Returns (micro_f1, macro_f1, avg_loss) on a DataLoader."""
    model.eval()
    all_probs, all_labels, total_loss, n_batches = [], [], 0.0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            logits         = model(input_ids=input_ids, attention_mask=attention_mask).logits
            total_loss    += criterion(logits, labels).item()
            n_batches     += 1
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds      = (all_probs >= THRESHOLD).astype(int)

    return (
        f1_score(all_labels, preds, average="micro", zero_division=0),
        f1_score(all_labels, preds, average="macro", zero_division=0),
        total_loss / n_batches,
    )

In [ ]:
BEST_CKPT      = "/content/best_roberta_checkpoint"
best_val_micro = 0.0

print("=" * 70)
print(f"{'Epoch':<7} {'Train Loss':<14} {'Val Loss':<12} {'Val µF1':<12} {'Val mF1':<12} Saved?")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(
            model(input_ids=input_ids, attention_mask=attention_mask).logits, labels
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

        if step % 50 == 0:
            print(f"  [E{epoch} step {step}/{len(train_loader)}] loss={loss.item():.4f}")

    val_micro, val_macro, val_loss = evaluate(model, val_loader)
    avg_train = train_loss / len(train_loader)

    saved = ""
    if val_micro > best_val_micro:
        best_val_micro = val_micro
        model.save_pretrained(BEST_CKPT)
        tokenizer.save_pretrained(BEST_CKPT)
        saved = "✓ best"

    print(f"{epoch:<7} {avg_train:<14.4f} {val_loss:<12.4f} {val_micro:<12.4f} {val_macro:<12.4f} {saved}")

print("=" * 70)
print(f"Best val micro F1: {best_val_micro:.4f}")

## ⑨ Final evaluation on test set

In [ ]:
# Load the best checkpoint
best_model = RobertaForSequenceClassification.from_pretrained(
    BEST_CKPT, num_labels=NUM_LABELS
).to(DEVICE)

test_micro, test_macro, test_loss = evaluate(best_model, test_loader)
print(f"Test Micro F1 : {test_micro:.4f}")
print(f"Test Macro F1 : {test_macro:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

In [ ]:
# ── Per-label classification report ─────────────────────────────────────────────────────
best_model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        logits = best_model(
            input_ids      = batch["input_ids"].to(DEVICE),
            attention_mask = batch["attention_mask"].to(DEVICE),
        ).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch["labels"].numpy())

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)
preds      = (all_probs >= THRESHOLD).astype(int)

print(classification_report(all_labels, preds, target_names=DISTORTION_LABELS, zero_division=0))

## ⑩ Sanity test on example sentences

In [ ]:
def predict(text, model, threshold=THRESHOLD):
    enc = tokenizer(
        text, max_length=MAX_LEN, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(
            model(input_ids=enc["input_ids"].to(DEVICE),
                  attention_mask=enc["attention_mask"].to(DEVICE)).logits
        ).squeeze().cpu().numpy()
    detected = [(DISTORTION_LABELS[i], round(float(probs[i]), 3))
                for i in range(NUM_LABELS) if probs[i] >= threshold]
    return sorted(detected, key=lambda x: x[1], reverse=True)


tests = [
    ("I failed the exam - I'm a complete failure at everything I do.",
     "All-or-nothing thinking"),
    ("I always mess up presentations. I'll probably bomb this one too.",
     "Overgeneralization + Fortune-telling"),
    ("She didn't reply to my message. She must hate me.",
     "Mind Reading"),
    ("If I make one mistake in this meeting, my entire career is over.",
     "Catastrophising"),
    ("I should be able to handle this stress without complaining.",
     "Should statements"),
    ("I went for a walk this morning and it was quite pleasant.",
     "No distortion"),
]

print("=" * 70)
print("SANITY TESTS")
print("=" * 70)
for text, expected in tests:
    result = predict(text, best_model)
    print(f"\nText    : {text}")
    print(f"Expected: {expected}")
    print(f"Got     : {result if result else 'No distortions detected'}")
print("=" * 70)

## ⑪ Download model directly to your computer

In [ ]:
from google.colab import files

# Zip the best checkpoint folder
ZIP_SRC  = BEST_CKPT                               # /content/best_roberta_checkpoint
ZIP_DEST = "/content/roberta_cognitive_distortion" # zip will be created at this path + .zip

shutil.make_archive(ZIP_DEST, "zip", ZIP_SRC)

zip_file = ZIP_DEST + ".zip"
size_mb  = os.path.getsize(zip_file) / (1024 ** 2)
print(f"Archive: {zip_file}  ({size_mb:.1f} MB)")
print("Starting download - check your browser downloads...")

files.download(zip_file)

## ⑫ Next steps after downloading

1. **Unzip** `roberta_cognitive_distortion.zip`

2. **Place the folder** in your local project:
   ```
   NLP_CBT/
   └── models/
       └── roberta_cognitive_distortion/
           ├── config.json
           ├── model.safetensors
           ├── tokenizer.json
           ├── tokenizer_config.json
           ├── vocab.json
           └── merges.txt
   ```

3. **Test locally:**
   ```bash
   uvicorn src.api.main:app --reload --port 8000
   ```

4. **Push to HuggingFace Space:**
   ```bash
   git add models/roberta_cognitive_distortion/
   git commit -m "feat: upgrade to fine-tuned RoBERTa model"
   git push space main
   ```

> **Note:** The zip is ~500 MB. Make sure `.gitattributes` already tracks `*.safetensors` via Git LFS (it does - we set this up during the HuggingFace deployment day).